In [ ]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [ ]:
if torch.backends.mps.is_available():
    print("Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).")
    mps_enabled = True
    torch.set_default_dtype(torch.float32)
    print("set default to float32")

In [ ]:
import keras
import tensorflow as tf

In [ ]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [ ]:
keras.backend.backend()

In [ ]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]
input_size = config["inputSize"]
output_size = config["outputSize"]
seed = config["seed"]

In [ ]:
keras.utils.set_random_seed(seed)

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
# Dataset initialization

from utils.data_loader import get_ml_cup_data, split_dataloader
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    match scaler_type:
        case "Standard":
            return StandardScaler()
        case "MinMax":
            return MinMaxScaler()
        case "Robust":
            return RobustScaler()
        case "MaxAbsScaler":
            return MaxAbsScaler()
        case _:
            return None

train_loader, test_loader = get_ml_cup_data(
    batch_size, 
    scaler=_scaler(),
    mps=mps_enabled
    )

In [ ]:
train_loader.dataset.X.shape, train_loader.dataset.y.shape

In [ ]:
test_loader.dataset.X.shape, test_loader.dataset.y.shape

In [ ]:
import numpy as np

y_mean = train_loader.dataset.y.mean(axis=0)        # (4,)

y_pred_baseline = np.tile(y_mean, (len(train_loader.dataset.y), 1))

mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)

mee_baseline = mee_errors.mean()
mse_baseline = mse_errors.mean()

print("Baseline MEE:", mee_baseline)
print("Baseline MSE:", mse_baseline)

In [ ]:
from keras import Sequential
from keras.layers import Input, Dense

In [ ]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee", dtype=torch.float32)

In [ ]:
from tensorflow import keras

def rs_build_model(
    learning_rate,
    units_1, units_2, units_3, units_4, units_5,
    lambda_1, lambda_2, lambda_3, lambda_4, lambda_5,
    activation_1="relu",
    activation_2="relu",
    activation_3="relu",
    activation_4="relu",
    activation_5="relu",
):
    """
    Build up to a 5-hidden-layer MLP.
    Any layer with units_i <= 0 is skipped.
    """

    inputs = keras.Input(shape=(input_size,))
    x = inputs

    # Layer 1 (required: you probably always give units_1 > 0)
    if units_1 > 0:
        x = keras.layers.Dense(
            units_1,
            activation=activation_1,
            kernel_regularizer=keras.regularizers.l2(lambda_1),
        )(x)

    # Layer 2 (optional)
    if units_2 > 0:
        x = keras.layers.Dense(
            units_2,
            activation=activation_2,
            kernel_regularizer=keras.regularizers.l2(lambda_2),
        )(x)

    # Layer 3 (optional)
    if units_3 > 0:
        x = keras.layers.Dense(
            units_3,
            activation=activation_3,
            kernel_regularizer=keras.regularizers.l2(lambda_3),
        )(x)

    # Layer 4 (optional)
    if units_4 > 0:
        x = keras.layers.Dense(
            units_4,
            activation=activation_4,
            kernel_regularizer=keras.regularizers.l2(lambda_4),
        )(x)

    # Layer 5 (optional)
    if units_5 > 0:
        x = keras.layers.Dense(
            units_5,
            activation=activation_5,
            kernel_regularizer=keras.regularizers.l2(lambda_5),
        )(x)

    # Output layer
    outputs = keras.layers.Dense(output_size)(x)

    model = keras.Model(inputs, outputs)

    optimizer = keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        clipnorm=1,
        nesterov=True,
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[mee],  # your existing custom metric
    )

    return model


In [ ]:
import utils.keras as ukeras

## Baseline optimizer (RandomSearch KFold CV)

In [ ]:
from scikeras.wrappers import KerasRegressor

early_stopping_cb = ukeras.make_early_stopping(mse_baseline, monitor="val_loss")
reg = KerasRegressor(
    model=rs_build_model,
    epochs=300,
    batch_size=batch_size,
    verbose=1,
    validation_split=.20,
    callbacks=[early_stopping_cb],
    loss="mean_squared_error",
    metrics=[mee]
)

In [ ]:
from scipy.stats import loguniform

param_distributions = {
    "model__learning_rate": loguniform(1e-4, 1e-2),
    "model__lambda_1": loguniform(1e-5, 1e-1),
    "model__lambda_2": loguniform(1e-5, 1e-1),
    "model__lambda_3": loguniform(1e-5, 1e-1),
    "model__lambda_4": loguniform(1e-5, 1e-1),
    "model__lambda_5": loguniform(1e-5, 1e-1),
    "model__activation_1": ["relu", "leaky_relu"],
    "model__activation_2": ["relu", "leaky_relu"],
    "model__activation_3": ["relu", "leaky_relu"],
    "model__activation_4": ["relu", "leaky_relu"],
    "model__activation_5": ["relu", "leaky_relu"],
    "model__units_1": [8, 16, 32],
    "model__units_2": [0, 4, 8, 16],
    "model__units_3": [0, 4, 8, 16],
    "model__units_4": [0, 4, 8, 16],
    "model__units_5": [0, 4, 8, 16],
}

In [ ]:
from sklearn.model_selection import KFold, RandomizedSearchCV

rs_basepath = "keras/models/rs"
k = 5
cv = KFold(n_splits=k, shuffle=True, random_state=seed)

random_search = RandomizedSearchCV(
    estimator=reg,
    param_distributions=param_distributions,
    n_iter=15,
    cv=cv,
    scoring="neg_mean_squared_error",
    verbose=1,
    random_state=seed,
)

# --- 6) Fit ---
if os.path.isdir(rs_basepath) and len(os.listdir(rs_basepath)) > 0:
    with open(rs_basepath + "/hp.json", "r") as f:
        rs_hp = json.load(f)
    print("hyperparameters loaded")
else:
    random_search.fit(train_loader.dataset.X, train_loader.dataset.y)
    rs_hp = random_search.best_params_
    print("Best CV MEE (negative):", random_search.best_score_)
    os.makedirs(rs_basepath, exist_ok=True)
    with open(rs_basepath + "/hp.json", "w") as f:
        json.dump(random_search.best_params_, f, indent=2)
        print("random_search hp saved")

print("Best params:", rs_hp)


In [ ]:
from keras.callbacks import TensorBoard

tensorboard_cb = TensorBoard(
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

In [ ]:
train_dataset = train_loader.dataset

In [ ]:
rs_model_basepath = "keras/models/rs"
if os.path.isfile(rs_model_basepath+"/model.keras"):
    rs_model = ukeras.load_saved_model(rs_model_basepath+"/model.keras")
    rs_model_regressor = KerasRegressor(rs_model)
    rs_model_regressor.model = rs_model
    print("RandomSearch model loaded")
else:
    print("RandomSearch hp loaded")
    os.makedirs(rs_model_basepath, exist_ok=True)
    early_stopping_cb = ukeras.make_early_stopping(mee_baseline, monitor="val_mee")
    early_stopping_cb.patience = 20
    tensorboard_cb.log_dir = ukeras.log_dir(ukeras.dict_to_filename(rs_hp, prefix="rs"))
    model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
        filepath=rs_model_basepath+"/model.keras",
        save_weights_only=False,
        monitor='val_mee',
        mode='min',
        save_best_only=True
    )
    rs_model_regressor = KerasRegressor(
        rs_build_model,
        batch_size=batch_size,
        epochs=500,
        verbose=1,
        validation_split=.20,
        loss="mean_squared_error",
        metrics=[mee],
        callbacks=[early_stopping_cb, model_checkpoint_callback]
    )
    rs_hp["model__learning_rate"] = 0.002
    rs_model_regressor.set_params(**rs_hp)
    rs_model_regressor.fit(
        X=train_dataset.X,
        y=train_dataset.y,
    )

In [ ]:
tr_mee_single, mee_single, tr_mse_single, mse_single = ukeras.assessment(
    model=rs_model,
    X_tr=train_loader.dataset.X,
    y_tr=train_loader.dataset.y,
    X_ts=test_loader.dataset.X,
    y_ts=test_loader.dataset.y,
    mee=mee
    )

if tr_mee_single < mee_single:
    print(f"overfitting mee_single {tr_mee_single - mee_single}")

if tr_mse_single < mse_single:
    print(f"overfitting mse_single {tr_mee_single - mse_single}")

errors_tr = ukeras.build_results_json(
    tr_mee_single, 
    0, 
    mee_baseline, 
    tr_mse_single, 
    0, 
    mse_baseline
)
errors_ts = ukeras.build_results_json(
    mee_single, 
    0, 
    mee_baseline, 
    mse_single, 
    0, 
    mse_baseline,
    prefix="ts",
    print_baseline=True
)

In [ ]:
errors_tr

In [ ]:
errors_ts

In [ ]:
with open("keras/assessment.json", "w+") as fp:
    print("writing assessment.json")
    json.dump({**errors_tr, **errors_ts}, fp, indent=2)

In [ ]:
from sklearn.metrics import PredictionErrorDisplay
from sklearn.model_selection import cross_val_predict

ukeras.plot_prediction_error(train_dataset.y, rs_model.predict(train_dataset.X))

In [ ]:
test_dataset = test_loader.dataset
ukeras.plot_prediction_error(test_dataset.y, rs_model.predict(test_dataset.X))

## Bayesian Optimization (Optuna-like model selection)

In [ ]:
from tensorflow import keras
from keras_tuner import HyperParameters

def bo_build_model(hp: HyperParameters):
    learning_rate = hp.Float(
        "learning_rate",
        min_value=1e-4,
        max_value=3e-3,
        sampling="log",
    )

    lambda_1 = hp.Float("lambda_1", 1e-5, 1e-1, sampling="log")
    lambda_2 = hp.Float("lambda_2", 1e-5, 1e-1, sampling="log")
    lambda_3 = hp.Float("lambda_3", 1e-5, 1e-1, sampling="log")
    lambda_4 = hp.Float("lambda_4", 1e-5, 1e-1, sampling="log")
    lambda_5 = hp.Float("lambda_5", 1e-5, 1e-1, sampling="log")

    activation_1 = hp.Choice("activation_1", ["relu", "leaky_relu"])
    activation_2 = hp.Choice("activation_2", ["relu", "leaky_relu"])
    activation_3 = hp.Choice("activation_3", ["relu", "leaky_relu"])
    activation_4 = hp.Choice("activation_4", ["relu", "leaky_relu"])
    activation_5 = hp.Choice("activation_5", ["relu", "leaky_relu"])

    units_1 = hp.Choice("units_1", [8, 16, 32])
    units_2 = hp.Choice("units_2", [0, 4, 8, 16])
    units_3 = hp.Choice("units_3", [0, 4, 8, 16])
    units_4 = hp.Choice("units_4", [0, 4, 8, 16])
    units_5 = hp.Choice("units_5", [0, 4, 8, 16])

    # ---- Model definition ----
    inputs = keras.Input(shape=(input_size,))
    x = inputs

    # Layer 1 (required)
    if units_1 > 0:
        x = keras.layers.Dense(
            units_1,
            activation=activation_1,
            kernel_regularizer=keras.regularizers.l2(lambda_1),
        )(x)

    # Layer 2
    if units_2 > 0:
        x = keras.layers.Dense(
            units_2,
            activation=activation_2,
            kernel_regularizer=keras.regularizers.l2(lambda_2),
        )(x)

    # Layer 3
    if units_3 > 0:
        x = keras.layers.Dense(
            units_3,
            activation=activation_3,
            kernel_regularizer=keras.regularizers.l2(lambda_3),
        )(x)

    # Layer 4
    if units_4 > 0:
        x = keras.layers.Dense(
            units_4,
            activation=activation_4,
            kernel_regularizer=keras.regularizers.l2(lambda_4),
        )(x)

    # Layer 5
    if units_5 > 0:
        x = keras.layers.Dense(
            units_5,
            activation=activation_5,
            kernel_regularizer=keras.regularizers.l2(lambda_5),
        )(x)

    outputs = keras.layers.Dense(output_size)(x)

    model = keras.Model(inputs, outputs)

    optimizer = keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        clipnorm=1.0,
        nesterov=True,
    )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[mee],   # your custom metric
    )

    return model


In [ ]:
import os
import json
import keras_tuner
from keras_tuner import BayesianOptimization

bo_basepath = "keras/models/bo"

max_trials = 20   # or whatever budget you want
executions_per_trial = 5  # you can set >1 to average noise if needed

# --- Early stopping callback (optional but recommended) ---
early_stopping_cb = ukeras.make_early_stopping(mse_baseline)

if os.path.isdir(bo_basepath) and len(os.listdir(bo_basepath)) > 0 \
   and os.path.exists(os.path.join(bo_basepath, "best5_hps/1_hp_config.json")) \
   and os.path.exists(os.path.join(bo_basepath, "best5_hps/1_hp.json")):

    # --- Load previously saved hyperparameters ---
    with open(os.path.join(bo_basepath, "best5_hps/1_hp.json"), "r") as f:
        bo_best_hp = json.load(f)

    print("BayesianOptimization hyperparameters loaded from disk.")

else:
    # --- Define tuner ---
    tuner = BayesianOptimization(
        hypermodel=bo_build_model,
        objective=keras_tuner.Objective("val_loss", direction="min"),
        max_trials=max_trials,
        executions_per_trial=executions_per_trial,
        directory=bo_basepath,
        project_name="bo_sgd_nn",
        seed=seed,
        overwrite=True
    )

    bo_train_loader, bo_val_loader = split_dataloader(train_loader, 0.2, batch_size, seed=seed)
    # --- Run search ---
    tuner.search(
        bo_train_loader,
        validation_data=bo_val_loader,
        epochs=300,
        callbacks=[early_stopping_cb],
        verbose=1,
    )

    # --- Extract best and top-5 hyperparameters ---
    best_5_hps= tuner.get_best_hyperparameters(5)
    for i, hp in enumerate(best_5_hps):
        ukeras.save_hyperparameters(hp, name=f"{i+1}_hp", dir=bo_basepath+"/best5_hps")

print("Best BO params:", bo_best_hp)

In [ ]:
from sklearn.model_selection import KFold
import numpy as np

bo_model_basepath = "keras/models/bo"

if os.path.isfile(bo_model_basepath + "/model.keras"):
    bo_model = ukeras.load_saved_model(bo_model_basepath + "/model.keras")
    print("Final BO model loaded from disk.")

    with open(bo_model_basepath + "/fold_mees.json", "r") as f:
        fold_mees = json.load(f)
    with open(bo_model_basepath + "/fold_mses.json", "r") as f:
        fold_mses = json.load(f)

    history = ukeras.load_history(bo_model_basepath)
else:
    # --- load best hyperparameters chosen by BO ---
    bo_best_hp = ukeras.load_hyperparameters(
        bo_model_basepath + "/best5_hps", "1_hp_config.json"
    )
    print("BO best hyperparameters loaded")

    # --- prepare data for KFold (using the same data RandomizedSearch used) ---
    X = train_loader.dataset.X   # shape (n_samples, n_features)
    y = train_loader.dataset.y   # shape (n_samples, n_targets)

    k = 5
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    fold_mees = []
    fold_mses = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"\n=== BO KFold: Fold {fold}/{k} ===")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # fresh callbacks for each fold
        early_stopping_cb = ukeras.make_early_stopping(
            mee_baseline, monitor="val_mee"
        )
        early_stopping_cb.patience = 20

        # (optional) per-fold TensorBoard/logging if you like
        # tensorboard_cb.log_dir = ukeras.log_dir(
        #     ukeras.dict_to_filename(bo_best_hp, prefix=f"bo_fold{fold}")
        # )

        # build a fresh model with fixed BO hyperparameters
        bo_model = bo_build_model(bo_best_hp)

        history = bo_model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=300,
            callbacks=[early_stopping_cb],
            verbose=0,
        )

        # best val_mee and val_mse for this fold
        val_mees = history.history["val_mee"]
        val_mses = history.history["val_loss"]  # loss="mse"
        best_val_mee = float(np.min(val_mees))
        best_val_mse = float(np.min(val_mses))

        fold_mees.append(best_val_mee)
        fold_mses.append(best_val_mse)

        print(f"Fold {fold}: best val_mee={best_val_mee:.4f}, val_mse={best_val_mse:.4f}")

    with open(bo_model_basepath + "/fold_mees.json", "w+") as f:
        json.dump(fold_mees, f, indent=2)
    with open(bo_model_basepath + "/fold_mses.json", "w+") as f:
        json.dump(fold_mses, f, indent=2)

    # --- overall BO CV performance ---
    mean_mee = float(np.mean(fold_mees))
    std_mee  = float(np.std(fold_mees))
    mean_mse = float(np.mean(fold_mses))
    std_mse  = float(np.std(fold_mses))

    print("\n=== BO KFold CV summary ===")
    print(f"MEE: mean={mean_mee:.4f}, std={std_mee:.4f}")
    print(f"MSE: mean={mean_mse:.4f}, std={std_mse:.4f}")

    early_stopping_cb = ukeras.make_early_stopping(
        mee_baseline, monitor="val_mee"
    )
    early_stopping_cb.patience = 20

    model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
        filepath=bo_model_basepath + "/model.keras",
        save_weights_only=False,
        monitor="val_mee",
        mode="min",
        save_best_only=True,
    )

    # small validation split just for early stopping on full data
    bo_model = bo_build_model(bo_best_hp)
    history = bo_model.fit(
        X,
        y,
        validation_split=0.2,
        epochs=300,
        callbacks=[early_stopping_cb, model_checkpoint_callback],
        verbose=1,
    )
    ukeras.save_history(history)
    print("History saved")

    print("Final BO model saved.")

In [ ]:
fold_mees,fold_mses

In [ ]:
ukeras.plot_cv_bar_per_fold(fold_mees, fold_mses, model_name="BO Best Model")

In [ ]:
ukeras.plot_loss_curve(history)

In [ ]:
ukeras.plot_loss_curve(history, log_y=True)

In [ ]:
tr_mee_single, mee_single, tr_mse_single, mse_single = ukeras.assessment(
    model=bo_model,
    X_tr=train_loader.dataset.X,
    y_tr=train_loader.dataset.y,
    X_ts=test_loader.dataset.X,
    y_ts=test_loader.dataset.y,
    mee=mee
    )

if tr_mee_single < mee_single:
    print(f"overfitting mee_single {tr_mee_single - mee_single}")

if tr_mse_single < mse_single:
    print(f"overfitting mse_single {tr_mee_single - mse_single}")

errors_tr = ukeras.build_results_json(
    tr_mee_single, 
    0, 
    mee_baseline, 
    tr_mse_single, 
    0, 
    mse_baseline
)
errors_ts = ukeras.build_results_json(
    mee_single, 
    0, 
    mee_baseline, 
    mse_single, 
    0, 
    mse_baseline,
    prefix="ts",
    print_baseline=True
)

In [ ]:
errors_tr

In [ ]:
errors_ts

In [ ]:
with open("keras/models/bo/assessment.json", "w+") as fp:
    print("writing assessment.json")
    json.dump({**errors_tr, **errors_ts}, fp, indent=2)

In [ ]:
ukeras.plot_prediction_error(train_dataset.y, bo_model.predict(train_dataset.X))

In [ ]:
test_dataset = test_loader.dataset
ukeras.plot_prediction_error(test_dataset.y, rs_model.predict(test_dataset.X))

In [ ]:
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

def log_dir(name, append:str=None):
    BASE = f"logs/{name}/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    if append:
        BASE += "_" + append
    return BASE

In [ ]:
# Early stopping
nn_es_enabled = config["nn"]["earlyStopping"]["enabled"]                ## true/false
nn_es_patience = config["nn"]["earlyStopping"]["patience"]              ## int
nn_es_mode = config["nn"]["earlyStopping"]["mode"]                      ## min/max
nn_es_restore_best = config["nn"]["earlyStopping"]["restoreBestWeight"] ## true/false

In [ ]:
tensorboard_cb = TensorBoard(
    log_dir=log_dir("fit"),
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

In [ ]:
tensorboard_cb.log_dir = log_dir("fit", nn_optimizer)

history = model.fit(
    train_loader, 
    validation_data=validation_loader, 
    epochs=nn_epochs,
    callbacks=[tensorboard_cb]
    )

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
# evaluate model
results = model.evaluate(test_loader, return_dict=True)
print(results)

In [ ]:
# write logs to a separate folder
writer = tf.summary.create_file_writer(log_dir("eval", nn_optimizer))

with writer.as_default():
    for k, v in results.items():
        tf.summary.scalar(k, v, step=0)

writer.close()

In [ ]:
%tensorboard --logdir logs/eval

In [ ]:
# --- Optuna ("optuna") section ---
optuna_epochs = config["optuna"]["epochs"]

# Suggestions for hidden layers
optuna_hidden_1_range = config["optuna"]["suggestions"]["hidden"][0]["range"]
optuna_hidden_1_activation = config["optuna"]["suggestions"]["hidden"][0]["activation"]

optuna_hidden_2_range = config["optuna"]["suggestions"]["hidden"][1]["range"]
optuna_hidden_2_activation = config["optuna"]["suggestions"]["hidden"][1]["activation"]

# Optuna early stopping
optuna_es_enabled = config["optuna"]["earlyStopping"]["enabled"]
optuna_es_patience = config["optuna"]["earlyStopping"]["patience"]
optuna_es_mode = config["optuna"]["earlyStopping"]["mode"]
optuna_es_restore_best = config["optuna"]["earlyStopping"]["restoreBestWeight"]

In [ ]:
## Optuna
import optuna
import tensorflow as tf

def objective(trial):
    # Suggest hyperparameters
    units1 = trial.suggest_int("units1", 16, 128)
    units2 = trial.suggest_int("units2", 16, 128)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Build model
    model = Sequential([
        Input(shape=(input_size,)),
        # Dense layers are fully connected layers
        Dense(units1, activation='relu'),
        Dense(units2, activation='relu'),
        Dense(output_size)
    ])

    optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss=mee
    )

    tensorboard_cb.log_dir = log_dir("fit", f"OPTUNA_TRIAL#{trial.number}")

    pruning_cb = optuna.integration.KerasPruningCallback(trial, "val_loss")
    
    checkpoint_path = f"checkpoints/optuna_trial_{trial.number}.keras"

    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=False
    )
    
    # Train model
    history = model.fit(
        train_loader,
        validation_data=validation_loader,
        epochs=optuna_epochs,
        verbose=0,
        callbacks=[tensorboard_cb, pruning_cb, checkpoint_cb]
    )

    val_loss = history.history["val_loss"][-1]
    return val_loss

In [ ]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=seed)
study = optuna.create_study(sampler=sampler, direction="minimize")
study.optimize(objective, n_trials=30)

print("Best trial:", study.best_trial.params)

In [ ]:
from utils.optuna import delete_pruned_trial_dirs

In [ ]:
delete_pruned_trial_dirs(root_path="logs/fit", study=study)

In [ ]:
study.best_trial

In [ ]:
best_trial = study.best_trial
best_model_path = f"checkpoints/optuna_trial_{best_trial.number}.keras"
best_model = keras.models.load_model(best_model_path)

In [ ]:
test_loss = best_model.evaluate(test_loader)

In [ ]:
writer = tf.summary.create_file_writer(log_dir("eval", f"OPTUNA_TRIAL#{best_trial.number}"))
with writer.as_default():
    tf.summary.scalar("loss", test_loss, step=0)
    writer.flush()

In [ ]:
%tensorboard --logdir logs/eval